# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the [FAIR^2 mass spectrometry protein abundance dataset](https://sen.science/doi/10.71728/senscience.vq4a-28xa) using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access dataset metadata (as an object, not a dict)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs using the dataset metadata.

In [ ]:
print("Record Sets in the dataset:")

# List all record sets and their @id and fields
record_sets = list(metadata.record_sets)
recordset_ids = [rs.id for rs in record_sets]
for rs in record_sets:
    print(f"- RecordSet @id: {rs.id}, Name: {rs.name}")
    print("    Fields:")
    for field in rs.fields:
        field_id = field.id
        field_name = getattr(field, 'name', field_id)
        print(f"       - Field @id: {field_id}, Name: {field_name}")
    print("")

### Previewing sample records from the main record set
Looping through a few example records to familiarize with the schema and values.

In [ ]:
# For demonstration, choose the first available record set
if recordset_ids:
    example_recordset_id = recordset_ids[0]
    print(f"Sample records from RecordSet @id={example_recordset_id}:")
    record_gen = dataset.records(record_set=example_recordset_id)
    for i, record in enumerate(record_gen):
        print(json.dumps(record, indent=2))
        if i >= 2:  # Show first 3 records only
            break
else:
    print("No record sets found in the schema.")

## 3. Data Extraction
Load data from specific record sets into DataFrames for analysis. Use the record set and field `@id`s from the overview above.

In [ ]:
# Load records from each record set into pandas DataFrames
dataframes = {}

for rs_id in recordset_ids:
    print(f"Loading records for RecordSet: {rs_id}")
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"- Columns for {rs_id}:\n  ", df.columns.tolist())
    print(df.head(2))
    print()

# For downstream analysis, pick the first record set for demonstration (update as needed)
main_recordset_id = recordset_ids[0] if recordset_ids else None

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
import numpy as np

if main_recordset_id:
    df = dataframes[main_recordset_id]
    # Print all fields to determine which is numeric
    print("Columns in the main DataFrame:", df.columns.tolist())

    # Guess candidate numeric fields (e.g., fields like 'MW', 'coverage', 'count')
    candidates = [col for col in df.columns if (df[col].dtype==np.float64 or df[col].dtype==np.int64)]
    if not candidates:
        # Try to infer numeric columns heuristically (attempt conversion)
        for col in df.columns:
            try:
                converted = pd.to_numeric(df[col])
                if not converted.isnull().all():
                    candidates.append(col)
            except Exception:
                continue
    print(f"Detected candidate numeric fields: {candidates}")

    # Choose one (update as appropriate for dataset; here we try 'MW' for molecular weight)
    for choice in ['MW', 'Molecular_Weight', 'molecular_weight', 'coverage', 'Coverage', 'count', 'Count']:
        if choice in df.columns:
            numeric_field = choice
            break
    else:
        numeric_field = candidates[0] if candidates else df.columns[0]

    print(f"Using numeric field for filtering: {numeric_field}")

    # Convert if not already numeric
    df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')

    # Filtering:
    threshold = df[numeric_field].quantile(0.90)  # Top 10% as example cut-off
    filtered_df = df[df[numeric_field] > threshold].copy()
    print(f"Filtered records with {numeric_field} > {threshold:.2f} (top 10%):")
    print(filtered_df.head())

    # Normalizing
    filtered_df[f"{numeric_field}_normalized"] = (
        (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    )
    print(f"Normalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Group by another key attribute if available
    for group_col in ['accession', 'description', 'protein', 'sample']:
        if group_col in df.columns:
            group_field = group_col
            break
    else:
        group_field = None

    if group_field is not None:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
        print(f"Grouped data by {group_field}:\n", grouped_df.head())
else:
    print("No main record set available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_recordset_id and numeric_field in df.columns:
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.show()

    # If a group_field is available and not too high-cardinality, do boxplot
    if group_field and df[group_field].nunique() < 30:
        plt.figure(figsize=(12,6))
        sns.boxplot(data=df, x=group_field, y=numeric_field)
        plt.title(f"{numeric_field} by {group_field}")
        plt.xticks(rotation=90)
        plt.show()
else:
    print("Skipping visualization: suitable numeric field or record set not detected.")

## 6. Conclusion
In this notebook, we've explored the FAIR^2 mass spectrometry dataset via its Croissant schema and the `mlcroissant` library. We've examined record sets and field IDs, loaded tabular data, performed basic exploratory analysis, and visualized key quantitative attributes.

**Key findings and observations:**
- Data is FAIR-friendly, fully self-describing (record sets, fields accessible via `@id`).
- Major numeric fields such as protein abundance or molecular weight (e.g., `MW`) can be filtered and normalized easily.
- Grouping by key attributes (like `accession` or `description`) enables comparative/statistical analysis across proteins or samples.
- The `mlcroissant` library streamlines extraction, integration, and transformation, making the dataset ready for downstream bioinformatics pipelines or machine learning tasks.